# 12.5_13 Cut-Point Investigation

This notebook inspects the saved artifacts for the `12.5_13` magnitude bin and helps answer:

- Are sources being recognized in the manifest?
- Are they being removed during pre-events tagging?
- Are cameras being excluded before or during `events.py`?
- Are many sources being routed into the periodic branch?
- Do they survive raw event detection?
- Which downstream filters are killing them?

Edit `RUN_DIR_OVERRIDE` in the next cell if you want to inspect a different `12.5_13` run directory.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (missing pyproject.toml).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())

MAG_BIN = "12.5_13"
MANIFEST_PATH = REPO_ROOT / "output" / "manifests-4-22-26" / f"lc_manifest_{MAG_BIN}.parquet"

# Change this if you want to inspect a specific run directory.
RUN_DIR_OVERRIDE = REPO_ROOT / "output" / "runs" / "output_bundle_12.5_13_home_bundle_12.5_13"


def find_candidate_runs(mag_bin):
    runs_root = REPO_ROOT / "output" / "runs"
    if not runs_root.exists():
        return []
    candidates = [path for path in runs_root.iterdir() if path.is_dir() and mag_bin in path.name]
    return sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)


def resolve_run_dir(mag_bin, override=None):
    if override is not None:
        override = Path(override)
        if override.exists():
            return override
    candidates = find_candidate_runs(mag_bin)
    return candidates[0] if candidates else None


def find_latest_match(pattern):
    matches = sorted(REPO_ROOT.glob(pattern), key=lambda path: path.stat().st_mtime, reverse=True)
    return matches[0] if matches else None


def resolve_artifact(run_dir, relative_path=None, global_pattern=None):
    candidates = []
    if run_dir is not None and relative_path is not None:
        candidates.append(Path(run_dir) / relative_path)
    if global_pattern is not None:
        fallback = find_latest_match(global_pattern)
        if fallback is not None:
            candidates.append(fallback)
    for path in candidates:
        if path.exists():
            return path
    return candidates[0] if candidates else None


def read_table(path, columns=None):
    if path is None or not Path(path).exists():
        return None
    path = Path(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        if columns is None:
            return pd.read_parquet(path)
        return pd.read_parquet(path, columns=columns)
    return pd.read_csv(path)


def get_table_columns(path):
    if path is None or not Path(path).exists():
        return []
    path = Path(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pq.ParquetFile(path).schema.names
    return list(pd.read_csv(path, nrows=0).columns)


def table_row_count(path):
    if path is None or not Path(path).exists():
        return None
    path = Path(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pq.ParquetFile(path).metadata.num_rows
    with path.open() as handle:
        n_lines = sum(1 for _ in handle)
    return max(n_lines - 1, 0)


def as_bool(series):
    if series is None:
        return None
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(bool)
    lowered = series.fillna("").astype(str).str.strip().str.lower()
    return lowered.isin({"1", "true", "t", "yes", "y"})


def nonempty_text(series):
    if series is None:
        return None
    return series.fillna("").astype(str).str.strip() != ""


def split_csv_tokens(series):
    if series is None:
        return pd.Series(dtype="object")
    tokens = (
        series.fillna("")
        .astype(str)
        .str.split(",")
        .explode()
        .astype(str)
        .str.strip()
    )
    return tokens[tokens != ""]


def first_present(columns, candidates):
    for column in candidates:
        if column in columns:
            return column
    return None


RUN_DIR = resolve_run_dir(MAG_BIN, RUN_DIR_OVERRIDE)
print(f"REPO_ROOT={REPO_ROOT}")
print(f"MAG_BIN={MAG_BIN}")
print(f"RUN_DIR={RUN_DIR}")


REPO_ROOT=/Users/calder/code/malca
MAG_BIN=12.5_13
RUN_DIR=/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13


In [2]:
candidate_runs = find_candidate_runs(MAG_BIN)
display(Markdown("## Candidate run directories"))
if candidate_runs:
    candidate_df = pd.DataFrame(
        {
            "run_dir": [str(path) for path in candidate_runs],
            "mtime": [pd.Timestamp(path.stat().st_mtime, unit="s") for path in candidate_runs],
        }
    )
    display(candidate_df)
else:
    print("No matching run directories found under output/runs.")

artifact_specs = [
    ("manifest", None, f"output/manifests-4-22-26/lc_manifest_{MAG_BIN}.parquet"),
    ("filtered_tag", f"tags/lc_filtered_{MAG_BIN}.parquet", f"output/runs/**/tags/lc_filtered_{MAG_BIN}.parquet"),
    ("rejected_tag", f"tags/rejected_tag_{MAG_BIN}.csv", f"output/runs/**/tags/rejected_tag_{MAG_BIN}.csv"),
    ("camera_medians", f"tags/camera_medians_{MAG_BIN}.parquet", f"output/runs/**/tags/camera_medians_{MAG_BIN}.parquet"),
    ("pregate", f"tags/pre_periodicity_{MAG_BIN}.parquet", f"output/runs/**/tags/pre_periodicity_{MAG_BIN}.parquet"),
    ("periodic_branch", f"manifests/lc_periodic_branch_{MAG_BIN}.parquet", f"output/runs/**/manifests/lc_periodic_branch_{MAG_BIN}.parquet"),
    ("stochastic_branch", f"manifests/lc_stochastic_branch_{MAG_BIN}.parquet", f"output/runs/**/manifests/lc_stochastic_branch_{MAG_BIN}.parquet"),
    ("raw_events", f"results/lc_events_results_{MAG_BIN}.parquet", f"output/runs/**/results/lc_events_results_{MAG_BIN}.parquet"),
    ("filtered_events", f"results/lc_events_filtered_{MAG_BIN}.parquet", f"output/runs/**/results/lc_events_filtered_{MAG_BIN}.parquet"),
]

ARTIFACTS = {}
artifact_rows = []
for label, relative_path, global_pattern in artifact_specs:
    path = resolve_artifact(RUN_DIR, relative_path=relative_path, global_pattern=global_pattern)
    ARTIFACTS[label] = path
    artifact_rows.append(
        {
            "artifact": label,
            "path": str(path) if path is not None else None,
            "exists": bool(path is not None and path.exists()),
            "rows_if_table": table_row_count(path),
        }
    )

display(Markdown("## Resolved artifacts"))
display(pd.DataFrame(artifact_rows))


## Candidate run directories

,run_dir,mtime
0,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13,2026-03-12 01:03:21.204871655


## Resolved artifacts

,artifact,path,exists,rows_if_table
0,manifest,/Users/calder/code/malca/output/manifests-4-22-26/lc_manifest_12.5_13.parquet,True,1443430.0
1,filtered_tag,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13/tags/lc_filtered_12.5_13.parquet,False,NaN
2,rejected_tag,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13/tags/rejected_tag_12.5_13.csv,False,NaN
3,camera_medians,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13/tags/camera_medians_12.5_13.parquet,False,NaN
4,pregate,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13/tags/pre_periodicity_12.5_13.parquet,False,NaN
5,periodic_branch,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13/manifests/lc_periodic_branch_12.5_13....,False,NaN
6,stochastic_branch,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13/manifests/lc_stochastic_branch_12.5_1...,False,NaN
7,raw_events,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13/results/lc_events_results_12.5_13.par...,True,1431962.0
8,filtered_events,/Users/calder/code/malca/output/runs/output_bundle_12.5_13_home_bundle_12.5_13/results/lc_events_filtered_12.5_13.pa...,True,1431958.0


In [3]:
display(Markdown("## Manifest existence"))

MANIFEST = read_table(ARTIFACTS["manifest"])
MANIFEST_DAT_EXISTS_MASK = None
MANIFEST_DAT_EXISTS_TRUE = None

if MANIFEST is None:
    print("Manifest file not found.")
else:
    summary_rows = [{"metric": "manifest_rows", "value": len(MANIFEST)}]
    if "source_id" in MANIFEST.columns:
        summary_rows.append({"metric": "manifest_unique_source_id", "value": MANIFEST["source_id"].nunique(dropna=True)})
    if "dat_exists" in MANIFEST.columns:
        MANIFEST_DAT_EXISTS_MASK = as_bool(MANIFEST["dat_exists"])
        MANIFEST_DAT_EXISTS_TRUE = int(MANIFEST_DAT_EXISTS_MASK.sum())
        summary_rows.append({"metric": "dat_exists_true", "value": MANIFEST_DAT_EXISTS_TRUE})
        summary_rows.append({"metric": "dat_exists_false", "value": int((~MANIFEST_DAT_EXISTS_MASK).sum())})
        summary_rows.append({"metric": "dat_exists_true_pct", "value": 100.0 * MANIFEST_DAT_EXISTS_TRUE / len(MANIFEST)})
    display(pd.DataFrame(summary_rows))

    if "index_num" in MANIFEST.columns and MANIFEST_DAT_EXISTS_MASK is not None:
        index_summary = (
            MANIFEST.assign(dat_exists_bool=MANIFEST_DAT_EXISTS_MASK.astype(int))
            .groupby("index_num", dropna=False)
            .agg(total_rows=("source_id", "size"), dat_exists_true=("dat_exists_bool", "sum"))
            .reset_index()
        )
        index_summary["missing_rows"] = index_summary["total_rows"] - index_summary["dat_exists_true"]
        index_summary["missing_pct"] = 100.0 * index_summary["missing_rows"] / index_summary["total_rows"]
        display(Markdown("### Missing rates by index_num"))
        display(index_summary.sort_values(["missing_pct", "missing_rows"], ascending=[False, False]).head(20))

    if MANIFEST_DAT_EXISTS_MASK is not None:
        sample_cols = [col for col in ["source_id", "index_num", "dat_path"] if col in MANIFEST.columns]
        missing_rows = MANIFEST.loc[~MANIFEST_DAT_EXISTS_MASK, sample_cols]
        display(Markdown("### Sample rows with dat_exists=False"))
        display(missing_rows.head(20))

        if "dat_path" in MANIFEST.columns:
            missing_lc_dirs = (
                MANIFEST.loc[~MANIFEST_DAT_EXISTS_MASK, "dat_path"]
                .astype(str)
                .str.extract(r"/(lc\d+_cal)/")[0]
                .value_counts()
                .rename_axis("lc_dir")
                .reset_index(name="missing_rows")
            )
            display(Markdown("### Missing rows concentrated in which lc*_cal directories?"))
            display(missing_lc_dirs.head(20))


## Manifest existence

,metric,value
0,manifest_rows,1.443430e+06
1,manifest_unique_source_id,1.443430e+06
2,dat_exists_true,1.432190e+06
3,dat_exists_false,1.124000e+04
4,dat_exists_true_pct,9.922130e+01


### Missing rates by index_num

,index_num,total_rows,dat_exists_true,missing_rows,missing_pct
14,14,69799,69223,576,0.825227
8,8,79672,79017,655,0.822121
15,15,74306,73701,605,0.814201
11,11,65932,65405,527,0.799308
10,10,68243,67698,545,0.798617
17,17,73877,73290,587,0.794564
18,18,78076,77461,615,0.787694
1,1,68318,67781,537,0.786030
16,16,75443,74850,593,0.786024
4,4,72838,72269,569,0.781186


### Sample rows with dat_exists=False

,source_id,index_num,dat_path
233,1021784,10,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc10_cal/1021784.dat2
375,103079215729,17,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc17_cal/103079215729.dat2
422,103079219519,4,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc4_cal/103079219519.dat2
509,103079226629,11,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc11_cal/103079226629.dat2
762,103079245875,16,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc16_cal/103079245875.dat2
838,103079252313,6,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc6_cal/103079252313.dat2
1132,103079277108,1,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc1_cal/103079277108.dat2
1273,103079286112,1,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc1_cal/103079286112.dat2
1414,103079295362,8,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc8_cal/103079295362.dat2
1475,103079299851,3,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc3_cal/103079299851.dat2


### Missing rows concentrated in which lc*_cal directories?

,lc_dir,missing_rows
0,lc8_cal,655
1,lc18_cal,615
2,lc15_cal,605
3,lc19_cal,604
4,lc7_cal,595
5,lc16_cal,593
6,lc17_cal,587
7,lc14_cal,576
8,lc4_cal,569
9,lc6_cal,549


In [4]:
display(Markdown("## Pre-events tag cuts"))

TAG_FILTERED = read_table(ARTIFACTS["filtered_tag"])
if TAG_FILTERED is None:
    print("Filtered tag parquet not found. This often means the selected run directory only has downstream results.")
else:
    tag_summary = [{"metric": "filtered_tag_rows", "value": len(TAG_FILTERED)}]
    if "source_id" in TAG_FILTERED.columns:
        tag_summary.append({"metric": "filtered_tag_unique_source_id", "value": TAG_FILTERED["source_id"].nunique(dropna=True)})
    if MANIFEST_DAT_EXISTS_TRUE is not None:
        tag_summary.append({"metric": "fraction_of_manifest_dat_exists_kept", "value": 100.0 * len(TAG_FILTERED) / MANIFEST_DAT_EXISTS_TRUE})
    display(pd.DataFrame(tag_summary))

    tag_failed_cols = [col for col in ["failed_sparse", "failed_multi_camera", "failed_mag_range", "failed_any"] if col in TAG_FILTERED.columns]
    if tag_failed_cols:
        tag_failed_summary = pd.DataFrame(
            {
                "failed_col": tag_failed_cols,
                "rows_true": [int(as_bool(TAG_FILTERED[col]).sum()) for col in tag_failed_cols],
            }
        )
        display(Markdown("### Failed_* counts inside the saved filtered tag parquet"))
        display(tag_failed_summary)
    else:
        print("No failed_* tag columns found in the saved filtered parquet.")

REJECTED_TAG = read_table(ARTIFACTS["rejected_tag"])
display(Markdown("## Rejected-tag log"))
if REJECTED_TAG is None:
    print("Rejected-tag CSV not found.")
else:
    rejected_id_col = first_present(REJECTED_TAG.columns, ["source_id", "asas_sn_id", "ASAS-SN ID", "path", "dat_path"])
    rejected_summary = [{"metric": "rejected_tag_rows", "value": len(REJECTED_TAG)}]
    if rejected_id_col is not None:
        rejected_summary.append({"metric": f"unique_{rejected_id_col}", "value": REJECTED_TAG[rejected_id_col].astype(str).nunique()})
    display(pd.DataFrame(rejected_summary))

    if "rejection_reason" in REJECTED_TAG.columns:
        reason_rows = REJECTED_TAG["rejection_reason"].value_counts(dropna=False).rename_axis("rejection_reason").reset_index(name="rows")
        if rejected_id_col is not None:
            unique_sources = (
                REJECTED_TAG.groupby("rejection_reason")[rejected_id_col]
                .nunique(dropna=True)
                .rename("unique_sources")
                .reset_index()
            )
            reason_rows = reason_rows.merge(unique_sources, on="rejection_reason", how="left")
        display(Markdown("### Rejection reasons"))
        display(reason_rows)

    stat_cols = [col for col in ["time_span_days", "points_per_day", "n_cameras"] if col in REJECTED_TAG.columns]
    if stat_cols and "rejection_reason" in REJECTED_TAG.columns:
        agg = REJECTED_TAG.groupby("rejection_reason")[stat_cols].agg(["median", "min", "max"])
        display(Markdown("### Tag-stat summaries by rejection reason"))
        display(agg)

    sample_cols = [col for col in [rejected_id_col, "rejection_reason", "time_span_days", "points_per_day", "n_cameras", "mag_bin", "path", "dat_path"] if col is not None and col in REJECTED_TAG.columns]
    display(Markdown("### Sample rejected rows"))
    display(REJECTED_TAG[sample_cols].head(20))


## Pre-events tag cuts

Filtered tag parquet not found. This often means the selected run directory only has downstream results.


## Rejected-tag log

Rejected-tag CSV not found.


In [5]:
display(Markdown("## Camera-median exclusions"))

CAMERA_MEDIANS = read_table(ARTIFACTS["camera_medians"])
if CAMERA_MEDIANS is None:
    print("camera_medians parquet not found.")
else:
    excluded_mask = nonempty_text(CAMERA_MEDIANS["excluded_cameras"]) if "excluded_cameras" in CAMERA_MEDIANS.columns else pd.Series(False, index=CAMERA_MEDIANS.index)
    excluded_counts = split_csv_tokens(CAMERA_MEDIANS["excluded_cameras"]).value_counts().rename_axis("camera_id").reset_index(name="rows") if "excluded_cameras" in CAMERA_MEDIANS.columns else pd.DataFrame()
    n_excluded_per_source = split_csv_tokens(CAMERA_MEDIANS["excluded_cameras"]).groupby(level=0).size() if "excluded_cameras" in CAMERA_MEDIANS.columns else pd.Series(dtype=int)

    camera_summary = [
        {"metric": "rows", "value": len(CAMERA_MEDIANS)},
        {"metric": "sources_with_excluded_cameras", "value": int(excluded_mask.sum())},
        {"metric": "sources_with_excluded_cameras_pct", "value": 100.0 * float(excluded_mask.sum()) / len(CAMERA_MEDIANS) if len(CAMERA_MEDIANS) else np.nan},
        {"metric": "median_n_excluded_cameras_per_flagged_source", "value": float(n_excluded_per_source.median()) if not n_excluded_per_source.empty else np.nan},
        {"metric": "max_n_excluded_cameras_per_source", "value": int(n_excluded_per_source.max()) if not n_excluded_per_source.empty else 0},
    ]
    display(pd.DataFrame(camera_summary))
    if not excluded_counts.empty:
        display(Markdown("### Most commonly excluded camera IDs"))
        display(excluded_counts.head(20))

display(Markdown("## Pre-periodicity routing"))

PREGATE = read_table(ARTIFACTS["pregate"])
if PREGATE is None:
    print("pre_periodicity parquet not found.")
else:
    pre_flag = as_bool(PREGATE["pre_periodic_flag"]) if "pre_periodic_flag" in PREGATE.columns else pd.Series(False, index=PREGATE.index)
    pregate_summary = [
        {"metric": "rows", "value": len(PREGATE)},
        {"metric": "pre_periodic_flag_true", "value": int(pre_flag.sum())},
        {"metric": "pre_periodic_flag_false", "value": int((~pre_flag).sum())},
        {"metric": "pre_periodic_flag_true_pct", "value": 100.0 * float(pre_flag.sum()) / len(PREGATE) if len(PREGATE) else np.nan},
        {"metric": "periodic_branch_rows", "value": table_row_count(ARTIFACTS["periodic_branch"])},
        {"metric": "stochastic_branch_rows", "value": table_row_count(ARTIFACTS["stochastic_branch"])},
    ]
    display(pd.DataFrame(pregate_summary))

    if "pre_periodicity_label" in PREGATE.columns:
        label_counts = PREGATE["pre_periodicity_label"].value_counts(dropna=False).rename_axis("pre_periodicity_label").reset_index(name="rows")
        display(Markdown("### Pre-periodicity labels"))
        display(label_counts)

    numeric_cols = [col for col in ["pre_n_points", "pre_n_cameras", "pre_periodicity_scatter_ratio", "pre_ce_snr"] if col in PREGATE.columns]
    if numeric_cols:
        numeric_summary = PREGATE.assign(pre_periodic_flag=pre_flag).groupby("pre_periodic_flag")[numeric_cols].agg(["median", "min", "max"])
        display(Markdown("### Pregate numeric summaries by branch"))
        display(numeric_summary)


## Camera-median exclusions

camera_medians parquet not found.


## Pre-periodicity routing

pre_periodicity parquet not found.


In [6]:
display(Markdown("## Raw event-detection output"))

raw_event_columns = [
    "path",
    "dip_significant",
    "jump_significant",
    "dip_count",
    "jump_count",
    "dip_run_count",
    "jump_run_count",
    "dip_max_run_points",
    "jump_max_run_points",
    "dip_max_run_cameras",
    "jump_max_run_cameras",
    "failed_signal_amplitude",
    "bad_cameras_filtered",
    "baseline_source",
]
raw_event_path = ARTIFACTS["raw_events"]
raw_event_cols_available = get_table_columns(raw_event_path)
RAW_EVENTS = read_table(raw_event_path, columns=[col for col in raw_event_columns if col in raw_event_cols_available])

if RAW_EVENTS is None:
    print("Raw event results parquet not found.")
else:
    dip_sig = as_bool(RAW_EVENTS["dip_significant"]) if "dip_significant" in RAW_EVENTS.columns else pd.Series(False, index=RAW_EVENTS.index)
    jump_sig = as_bool(RAW_EVENTS["jump_significant"]) if "jump_significant" in RAW_EVENTS.columns else pd.Series(False, index=RAW_EVENTS.index)
    sig_any = dip_sig | jump_sig
    amp_fail = as_bool(RAW_EVENTS["failed_signal_amplitude"]) if "failed_signal_amplitude" in RAW_EVENTS.columns else pd.Series(False, index=RAW_EVENTS.index)
    bad_cam_filtered = nonempty_text(RAW_EVENTS["bad_cameras_filtered"]) if "bad_cameras_filtered" in RAW_EVENTS.columns else pd.Series(False, index=RAW_EVENTS.index)
    phase_fallback = RAW_EVENTS["baseline_source"].fillna("").astype(str).str.contains("phase_template_fallback") if "baseline_source" in RAW_EVENTS.columns else pd.Series(False, index=RAW_EVENTS.index)
    run_cameras_ge_2 = (
        (RAW_EVENTS.get("dip_max_run_cameras", 0).fillna(0) >= 2)
        | (RAW_EVENTS.get("jump_max_run_cameras", 0).fillna(0) >= 2)
    ) if {"dip_max_run_cameras", "jump_max_run_cameras"}.issubset(RAW_EVENTS.columns) else pd.Series(False, index=RAW_EVENTS.index)
    run_points_ge_2 = (
        (RAW_EVENTS.get("dip_max_run_points", 0).fillna(0) >= 2)
        | (RAW_EVENTS.get("jump_max_run_points", 0).fillna(0) >= 2)
    ) if {"dip_max_run_points", "jump_max_run_points"}.issubset(RAW_EVENTS.columns) else pd.Series(False, index=RAW_EVENTS.index)

    raw_summary = pd.DataFrame(
        [
            {"metric": "rows", "value": len(RAW_EVENTS)},
            {"metric": "dip_significant", "value": int(dip_sig.sum())},
            {"metric": "jump_significant", "value": int(jump_sig.sum())},
            {"metric": "significant_any", "value": int(sig_any.sum())},
            {"metric": "failed_signal_amplitude", "value": int(amp_fail.sum())},
            {"metric": "significant_and_failed_signal_amplitude", "value": int((sig_any & amp_fail).sum())},
            {"metric": "rows_with_bad_cameras_filtered", "value": int(bad_cam_filtered.sum())},
            {"metric": "significant_and_bad_cameras_filtered", "value": int((sig_any & bad_cam_filtered).sum())},
            {"metric": "significant_and_run_points_ge_2", "value": int((sig_any & run_points_ge_2).sum())},
            {"metric": "significant_and_run_cameras_ge_2", "value": int((sig_any & run_cameras_ge_2).sum())},
            {"metric": "significant_and_only_one_camera_run", "value": int((sig_any & ~run_cameras_ge_2).sum())},
            {"metric": "rows_with_phase_template_fallback", "value": int(phase_fallback.sum())},
            {"metric": "significant_and_phase_template_fallback", "value": int((sig_any & phase_fallback).sum())},
        ]
    )
    display(raw_summary)

    if "baseline_source" in RAW_EVENTS.columns:
        baseline_counts = RAW_EVENTS["baseline_source"].fillna("<missing>").astype(str).value_counts(dropna=False).rename_axis("baseline_source").reset_index(name="rows")
        display(Markdown("### baseline_source counts"))
        display(baseline_counts)

    if "bad_cameras_filtered" in RAW_EVENTS.columns:
        bad_camera_counts = split_csv_tokens(RAW_EVENTS["bad_cameras_filtered"]).value_counts().rename_axis("camera_id").reset_index(name="rows")
        if not bad_camera_counts.empty:
            display(Markdown("### Most commonly auto-filtered camera IDs inside events.py"))
            display(bad_camera_counts.head(20))

    sample_cols = [
        col for col in [
            "path",
            "dip_significant",
            "jump_significant",
            "dip_max_run_points",
            "jump_max_run_points",
            "dip_max_run_cameras",
            "jump_max_run_cameras",
            "failed_signal_amplitude",
            "bad_cameras_filtered",
            "baseline_source",
        ] if col in RAW_EVENTS.columns
    ]
    interesting = RAW_EVENTS.loc[sig_any & (~run_cameras_ge_2 | amp_fail | bad_cam_filtered | phase_fallback), sample_cols]
    display(Markdown("### Sample significant rows that still look fragile"))
    display(interesting.head(20))


## Raw event-detection output

,metric,value
0,rows,1431962
1,dip_significant,42800
2,jump_significant,34882
3,significant_any,66221
4,failed_signal_amplitude,1400271
5,significant_and_failed_signal_amplitude,51438
6,rows_with_bad_cameras_filtered,131475
7,significant_and_bad_cameras_filtered,17528
8,significant_and_run_points_ge_2,66221
9,significant_and_run_cameras_ge_2,14926


### baseline_source counts

,baseline_source,rows
0,gp_sho,1388745
1,"gp_sho,median_fallback",43216
2,median_fallback,1


### Most commonly auto-filtered camera IDs inside events.py

,camera_id,rows
0,1,26842
1,2,26550
2,3,24944
3,4,20692
4,5,16281
5,6,13812
6,7,7844
7,8,4449
8,9,2830
9,10,2011


### Sample significant rows that still look fragile

,path,dip_significant,jump_significant,dip_max_run_points,jump_max_run_points,dip_max_run_cameras,jump_max_run_cameras,failed_signal_amplitude,bad_cameras_filtered,baseline_source
66,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc19_cal/1005386.dat2,True,True,3,3,1,1,True,"1,5",gp_sho
138,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc15_cal/1005647.dat2,True,True,5,6,1,1,False,,gp_sho
174,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc15_cal/1015933.dat2,True,False,2,0,1,0,True,,gp_sho
281,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc5_cal/1024802.dat2,False,True,0,2,0,1,True,,gp_sho
509,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc4_cal/103079226646.dat2,True,False,2,0,1,0,True,,gp_sho
519,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc17_cal/103079226625.dat2,True,True,5,2,1,1,False,2,gp_sho
817,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc6_cal/103079249846.dat2,True,False,3,0,1,0,True,,gp_sho
848,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc18_cal/103079253266.dat2,False,True,0,6,0,2,True,,gp_sho
938,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc12_cal/103079260468.dat2,True,False,2,0,1,0,False,,gp_sho
1202,/data/poohbah/1/assassin/rowan.90/lcsv2/12.5_13/lc1_cal/103079282054.dat2,True,False,2,0,1,0,True,,gp_sho


In [7]:
display(Markdown("## Post-filter output"))

filtered_event_path = ARTIFACTS["filtered_events"]
filtered_event_cols = get_table_columns(filtered_event_path)
failure_cols = sorted(col for col in filtered_event_cols if col.startswith("failed_"))
FILTERED_EVENTS = read_table(filtered_event_path, columns=[col for col in (["path"] + failure_cols) if col in filtered_event_cols])

if FILTERED_EVENTS is None:
    print("Filtered event parquet not found.")
else:
    filtered_summary = [{"metric": "rows", "value": len(FILTERED_EVENTS)}]
    if "failed_any" in FILTERED_EVENTS.columns:
        failed_any_mask = as_bool(FILTERED_EVENTS["failed_any"])
        filtered_summary.append({"metric": "passed_all_filters", "value": int((~failed_any_mask).sum())})
        filtered_summary.append({"metric": "failed_any", "value": int(failed_any_mask.sum())})
        filtered_summary.append({"metric": "passed_all_filters_pct", "value": 100.0 * float((~failed_any_mask).sum()) / len(FILTERED_EVENTS) if len(FILTERED_EVENTS) else np.nan})
    display(pd.DataFrame(filtered_summary))

    if failure_cols:
        failed_counts = pd.DataFrame(
            {
                "failed_col": failure_cols,
                "rows_true": [int(as_bool(FILTERED_EVENTS[col]).sum()) for col in failure_cols],
            }
        ).sort_values("rows_true", ascending=False)
        display(Markdown("### Downstream failed_* counts"))
        display(failed_counts)
    else:
        print("No failed_* columns found in the filtered events parquet.")

display(Markdown("## Combined stage counts"))

waterfall_rows = []
if MANIFEST is not None:
    waterfall_rows.append({"stage": "manifest_rows", "rows": len(MANIFEST)})
if MANIFEST_DAT_EXISTS_TRUE is not None:
    waterfall_rows.append({"stage": "manifest_dat_exists_true", "rows": MANIFEST_DAT_EXISTS_TRUE})
if TAG_FILTERED is not None:
    waterfall_rows.append({"stage": "filtered_tag_rows", "rows": len(TAG_FILTERED)})
if CAMERA_MEDIANS is not None and "excluded_cameras" in CAMERA_MEDIANS.columns:
    waterfall_rows.append({"stage": "camera_median_nonempty_exclusions", "rows": int(nonempty_text(CAMERA_MEDIANS["excluded_cameras"]).sum())})
if PREGATE is not None and "pre_periodic_flag" in PREGATE.columns:
    pregate_flag = as_bool(PREGATE["pre_periodic_flag"])
    waterfall_rows.append({"stage": "pregate_periodic_branch", "rows": int(pregate_flag.sum())})
    waterfall_rows.append({"stage": "pregate_stochastic_branch", "rows": int((~pregate_flag).sum())})
if RAW_EVENTS is not None:
    raw_sig = as_bool(RAW_EVENTS["dip_significant"]) | as_bool(RAW_EVENTS["jump_significant"]) if {"dip_significant", "jump_significant"}.issubset(RAW_EVENTS.columns) else pd.Series(False, index=RAW_EVENTS.index)
    waterfall_rows.append({"stage": "raw_event_rows", "rows": len(RAW_EVENTS)})
    waterfall_rows.append({"stage": "raw_significant_any", "rows": int(raw_sig.sum())})
if FILTERED_EVENTS is not None:
    waterfall_rows.append({"stage": "filtered_event_rows", "rows": len(FILTERED_EVENTS)})
    if "failed_any" in FILTERED_EVENTS.columns:
        failed_any_mask = as_bool(FILTERED_EVENTS["failed_any"])
        waterfall_rows.append({"stage": "filtered_event_passed_all", "rows": int((~failed_any_mask).sum())})

if waterfall_rows:
    display(pd.DataFrame(waterfall_rows))
else:
    print("No stages were available to summarize.")


## Post-filter output

,metric,value
0,rows,1.431958e+06
1,passed_all_filters,6.074000e+03
2,failed_any,1.425884e+06
3,passed_all_filters_pct,4.241745e-01


### Downstream failed_* counts

,failed_col,rows_true
0,failed_any,1425884
2,failed_run_robustness,1418142
4,failed_signal_amplitude,1400043
1,failed_posterior_strength,413187
3,failed_score,15825


## Combined stage counts

,stage,rows
0,manifest_rows,1443430
1,manifest_dat_exists_true,1432190
2,raw_event_rows,1431962
3,raw_significant_any,66221
4,filtered_event_rows,1431958
5,filtered_event_passed_all,6074
